# Task manager with Tkinter

This lab builds a complete desktop application, a task manager: an entry and a button to add a task, the list of tasks, buttons to mark them done or remove them, and a filter (all, to do, done). It follows the course's Model-View-Controller organization, in four files the cells write: the model in `model.py`, the view in `view.py`, the controller in `controller.py`, which doesn't import tkinter, and the launch in `app.py`. The application runs outside the notebook, in its own window.

Showing it takes a screen:

- **on your machine**, in Jupyter or VS Code, with a Python that has Tk: `python -m tkinter` should open a small window (on Linux, otherwise install `python3-tk` with the package manager). From Colab, *File › Download › Download .ipynb* gets this notebook;
- **in the Codespace**, windows open on its desktop: in the *Ports* tab, next to the terminal, hover over the port 6080 line, *Bureau / Desktop*, and click the globe icon, *Open in Browser*, then *Connect* in the tab that opens, with no password;
- **in Colab**, there is no screen: the model and the controller can be written and tested, but the windows don't open.

This is the reference lab: do it on your machine or in the Codespace. The twin lab, the same application with ipywidgets, comes next and only rewrites the view; failing any other environment, it runs in full in Colab.

This is a linear lab: each part builds on the previous one. If you get stuck, run the solution and move on.

## Opening a window from the notebook

The `launch` function runs Python in a separate process, like `python app.py` in a terminal: the cell keeps running while the window is open, then shows what the program wrote (its `print`s, or the error that stopped it). Without a screen, it says so and launches nothing.

The last line opens Tk's test window: close it to give control back to the notebook.

In [ ]:
import subprocess
import sys


def launch(*arguments: str) -> None:
  screen = subprocess.run(
    [sys.executable, "-c", "import tkinter; tkinter.Tk().destroy()"],
    capture_output=True,
    text=True,
  )
  if screen.returncode != 0:
    cause = screen.stderr.strip().splitlines()[-1]
    print(f"No screen here, nothing is launched ({cause}).")
    return
  result = subprocess.run([sys.executable, *arguments], capture_output=True, text=True)
  print(result.stdout, result.stderr, sep="")


launch("-m", "tkinter")

## 1. The model

Write `model.py`, plain Python with no interface: it holds the application's rules.

- `FILTERS = ("all", "to do", "done")`, the filters the interface will offer.
- `Task`, a dataclass: a `title` (`str`) and `done` (`bool`, `False` by default). The title loses its leading and trailing spaces, and an empty title raises a `ValueError`.
- `Checklist`, a dataclass whose only attribute, `tasks`, is a list of `Task`, empty at first, with five methods:
  - `add(title)` creates the task, appends it to the list and returns it; a title already there raises a `ValueError`;
  - `remove(task)` takes it out of the list;
  - `toggle(task)` marks it done if it was to do, and the other way round;
  - `remove_done()` takes every done task out of the list;
  - `filtered(which)` returns the list of the tasks that match one of the `FILTERS`, in the order they were added; any other filter raises a `ValueError`.

The `ValueError` messages are meant for the user: the interface will show them as they are.

The cell starts with `%%writefile model.py`: Jupyter writes the rest of the cell to that file instead of running it. The check cell, after the solution, tests your model.

**Useful documentation**

- [%%writefile](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile)

In [ ]:
# Your code here

### Additional hints

The course on data classes shows the two tools `Task` and `Checklist` need: `__post_init__()`, which the dataclass calls after its generated `__init__`, validates the title (`self.title = self.title.strip()`, then `if not self.title: ...`); `field(default_factory=list)` gives each `Checklist` its own list. `any(...)` over a generator tells whether a task already has that title; compare with the cleaned-up title, `"  Bread "` is a duplicate of `"Bread"`. `toggle` is one line: `task.done = not task.done`, and so is `remove_done`: a list comprehension of the tasks to do.

```python
%%writefile model.py
from dataclasses import dataclass, field

FILTERS = ("all", "to do", "done")


@dataclass
class Task:
  title: str
  done: bool = False

  def __post_init__(self) -> None:
    self.title = self.title.strip()
    if not self.title:
      msg = "a task needs a title"
      raise ValueError(msg)


@dataclass
class Checklist:
  tasks: list[Task] = field(default_factory=list)

  def add(self, title: str) -> Task:
    task = Task(title)
    if any(...):
      msg = f"“{task.title}” is already in the list"
      raise ValueError(msg)
    ...

  def remove(self, task: Task) -> None: ...

  def toggle(self, task: Task) -> None: ...

  def remove_done(self) -> None: ...

  def filtered(self, which: str) -> list[Task]:
    if which == "all":
      return list(self.tasks)
    ...
```

### Solution

In [ ]:
%%writefile model.py
from dataclasses import dataclass, field

FILTERS = ("all", "to do", "done")


@dataclass
class Task:
  title: str
  done: bool = False

  def __post_init__(self) -> None:
    self.title = self.title.strip()
    if not self.title:
      msg = "a task needs a title"
      raise ValueError(msg)


@dataclass
class Checklist:
  tasks: list[Task] = field(default_factory=list)

  def add(self, title: str) -> Task:
    task = Task(title)
    if any(other.title == task.title for other in self.tasks):
      msg = f"“{task.title}” is already in the list"
      raise ValueError(msg)
    self.tasks.append(task)
    return task

  def remove(self, task: Task) -> None:
    self.tasks.remove(task)

  def toggle(self, task: Task) -> None:
    task.done = not task.done

  def remove_done(self) -> None:
    self.tasks = [task for task in self.tasks if not task.done]

  def filtered(self, which: str) -> list[Task]:
    if which == "all":
      return list(self.tasks)
    if which == "to do":
      return [task for task in self.tasks if not task.done]
    if which == "done":
      return [task for task in self.tasks if task.done]
    msg = f"unknown filter: {which!r}"
    raise ValueError(msg)

### Check

The next cell reloads `model.py` (otherwise the notebook keeps the version it imported first), then plays a short scenario. Expected results: `['Buy bread']` to do, `['Review the course']` done, then three `ValueError`s (empty title, duplicate, unknown filter), then `['Buy bread']` after the removal, and `['Buy bread']` again after `remove_done()`, which takes out "Water the plants", marked done.

In [ ]:
import importlib

import model

importlib.reload(model)

checklist = model.Checklist()
checklist.add("Buy bread")
review = checklist.add("  Review the course ")
checklist.toggle(review)
print("To do:", [task.title for task in checklist.filtered("to do")])
print("Done:", [task.title for task in checklist.filtered("done")])

for attempt in (
  lambda: checklist.add("   "),
  lambda: checklist.add("Buy bread"),
  lambda: checklist.filtered("urgent"),
):
  try:
    attempt()
  except ValueError as error:
    print("ValueError:", error)
  else:
    print("No ValueError: to fix")

checklist.remove(review)
print("All:", [task.title for task in checklist.filtered("all")])

checklist.toggle(checklist.add("Water the plants"))
checklist.remove_done()
print("After remove_done:", [task.title for task in checklist.tasks])

## 2. The view

Write `view.py`: a `TaskView` class that inherits from `ttk.Frame`, like the course's `CounterView`. It knows nothing of the model: all it imports from `model.py` is `FILTERS`. It creates its widgets and places them with `grid`:

| Row | Widgets |
|---|---|
| 0 | `entry`, a `ttk.Entry` bound to `title`, a `tk.StringVar`; the `add` button ("Add") |
| 1 | `tasks`, a `tk.Listbox` where several tasks can be selected; on its right, a vertical scrollbar (`ttk.Scrollbar`) |
| 2 | across both columns, a `ttk.Frame` holding, on the left, the `toggle` ("Done / to do") and `remove` ("Remove") buttons and, on the right, `filter_choices`, a list of three `ttk.Radiobutton`s, one per filter of `FILTERS`, bound to `which`, a `tk.StringVar` set to `"all"` at first |
| 3 | `status`, a `ttk.Label` |

The window also has a menu bar: "File" › "Quit" (Ctrl+Q shortcut) closes the window; "Tasks", `tasks_menu`, will get the "Remove done tasks" entry from the controller.

When the window grows, the entry and the list widen, and the list grows in height. These widgets are the view's business: the controller won't touch them, it will only use the view's methods.

The first methods read or change the display:

- `show(lines, status)` replaces the list's content with `lines`, a list of strings, and `status`'s text with `status`;
- `selection()` returns the list of the selected lines' indices;
- `typed_title()` returns the entry's text, `clear_entry()` clears it, and `chosen_filter()` returns the chosen filter;
- `report(message)` shows the message in an error box.

The others connect the widgets to an action, a function without arguments that the controller will give them, like `on_plus(action)` in the course's `CounterView`:

- `on_add(action)`: the "Add" button and the Return key in the entry;
- `on_toggle(action)`: the "Done / to do" button and double-clicking a task;
- `on_remove(action)`: the "Remove" button;
- `on_filter(action)`: choosing a filter;
- `on_remove_done(action)`: the "Remove done tasks" entry of the "Tasks" menu.

The cells after the solution open a preview of the view, with two sample lines: its buttons do nothing yet.

**Useful documentation**

- [Listbox and Scrollbar (TkDocs)](https://tkdocs.com/tutorial/morewidgets.html#listbox)
- [Radiobutton (TkDocs)](https://tkdocs.com/tutorial/widgets.html#radiobutton)
- [Menus (TkDocs)](https://tkdocs.com/tutorial/menus.html)

In [ ]:
# Your code here

### Additional hints

- `tk.Listbox(self, height=10, selectmode="extended")`: Shift+click and Ctrl+click select several lines.
- The bar is linked both ways: `bar = ttk.Scrollbar(self, orient="vertical", command=self.tasks.yview)`, then `self.tasks.configure(yscrollcommand=bar.set)`. Placed with `sticky="nsw"`, it takes the list's height.
- The list fills its cell with `sticky="nsew"`; `self.columnconfigure(0, weight=1)` and `self.rowconfigure(1, weight=1)` give the extra space to the column of the entry and the list, and to the list's row.
- The three `ttk.Radiobutton(actions, text=name, variable=self.which, value=name)` share `self.which`: choosing one unchecks the others.
- In row 2's `ttk.Frame`, `pack(side="left")` and `pack(side="right")`: another container, so another geometry manager than `grid`. `pack(side="right")` lays out from right to left: place the radio buttons in reverse order, `reversed(self.filter_choices)`.
- Margins: `child.grid_configure(padx=5, pady=5)` for each child of `self.winfo_children()`.
- Menus go on the window, which the view finds with `self.winfo_toplevel()`: `tk.Menu(window)` for the bar, one `tk.Menu(menubar, tearoff=False)` per menu, `add_cascade(label=..., menu=...)`, `add_command(label=..., command=..., accelerator="Ctrl+Q")`, then `window.configure(menu=menubar)`. `accelerator=` shows the shortcut without creating it: `window.bind("<Control-q>", ...)` creates it.
- `insert("end", *lines)` inserts all the lines at once; `curselection()` returns a tuple of indices.
- A button, like a radio button, receives the action without calling it: `self.add.configure(command=action)`. A `bind` callback receives the event, which the action doesn't need: `self.entry.bind("<Return>", lambda event: action())`. A double-click is `"<Double-Button-1>"`. `on_remove_done` adds the entry to the menu: `self.tasks_menu.add_command(label="Remove done tasks", command=action)`.

```python
%%writefile view.py
import tkinter as tk
from collections.abc import Callable
from tkinter import messagebox, ttk

from model import FILTERS


class TaskView(ttk.Frame):
  def __init__(self, parent) -> None:
    super().__init__(parent, padding=10)
    self.title = tk.StringVar()
    self.entry = ttk.Entry(self, textvariable=self.title)
    self.entry.grid(row=0, column=0, sticky="ew")
    self.add = ttk.Button(self, text="Add")
    self.add.grid(row=0, column=1)
    self.tasks = ...
    bar = ...
    actions = ttk.Frame(self)
    actions.grid(row=2, column=0, columnspan=2, sticky="ew")
    self.toggle = ttk.Button(actions, text="Done / to do")
    self.toggle.pack(side="left")
    ...
    self.which = tk.StringVar(value=FILTERS[0])
    self.filter_choices = [...]
    ...
    self.status = ttk.Label(self)
    self.status.grid(row=3, column=0, columnspan=2, sticky="w")
    self.columnconfigure(0, weight=1)
    self.rowconfigure(1, weight=1)
    for child in self.winfo_children():
      ...
    self.create_menus()

  def create_menus(self) -> None:
    window = self.winfo_toplevel()
    menubar = tk.Menu(window)
    ...
    self.tasks_menu = tk.Menu(menubar, tearoff=False)
    menubar.add_cascade(label="Tasks", menu=self.tasks_menu)
    window.configure(menu=menubar)
    ...

  def show(self, lines: list[str], status: str) -> None:
    self.tasks.delete(0, "end")
    ...

  def selection(self) -> list[int]:
    return ...

  def typed_title(self) -> str:
    return self.title.get()

  def clear_entry(self) -> None: ...

  def chosen_filter(self) -> str: ...

  def report(self, message: str) -> None:
    messagebox.showerror("Tasks", message)

  def on_add(self, action: Callable[[], None]) -> None:
    self.add.configure(command=action)
    self.entry.bind("<Return>", lambda event: action())

  def on_toggle(self, action: Callable[[], None]) -> None: ...

  def on_remove(self, action: Callable[[], None]) -> None: ...

  def on_filter(self, action: Callable[[], None]) -> None: ...

  def on_remove_done(self, action: Callable[[], None]) -> None: ...
```

### Solution

In [ ]:
%%writefile view.py
import tkinter as tk
from collections.abc import Callable
from tkinter import messagebox, ttk

from model import FILTERS


class TaskView(ttk.Frame):
  def __init__(self, parent) -> None:
    super().__init__(parent, padding=10)
    self.title = tk.StringVar()
    self.entry = ttk.Entry(self, textvariable=self.title)
    self.entry.grid(row=0, column=0, sticky="ew")
    self.add = ttk.Button(self, text="Add")
    self.add.grid(row=0, column=1)
    self.tasks = tk.Listbox(self, height=10, selectmode="extended")
    self.tasks.grid(row=1, column=0, sticky="nsew")
    bar = ttk.Scrollbar(self, orient="vertical", command=self.tasks.yview)
    bar.grid(row=1, column=1, sticky="nsw")
    self.tasks.configure(yscrollcommand=bar.set)
    actions = ttk.Frame(self)
    actions.grid(row=2, column=0, columnspan=2, sticky="ew")
    self.toggle = ttk.Button(actions, text="Done / to do")
    self.toggle.pack(side="left")
    self.remove = ttk.Button(actions, text="Remove")
    self.remove.pack(side="left")
    self.which = tk.StringVar(value=FILTERS[0])
    self.filter_choices = [
      ttk.Radiobutton(actions, text=name, variable=self.which, value=name)
      for name in FILTERS
    ]
    for choice in reversed(self.filter_choices):
      choice.pack(side="right")
    self.status = ttk.Label(self)
    self.status.grid(row=3, column=0, columnspan=2, sticky="w")
    self.columnconfigure(0, weight=1)
    self.rowconfigure(1, weight=1)
    for child in self.winfo_children():
      child.grid_configure(padx=5, pady=5)
    self.create_menus()

  def create_menus(self) -> None:
    window = self.winfo_toplevel()
    menubar = tk.Menu(window)
    file_menu = tk.Menu(menubar, tearoff=False)
    file_menu.add_command(label="Quit", command=window.destroy, accelerator="Ctrl+Q")
    menubar.add_cascade(label="File", menu=file_menu)
    self.tasks_menu = tk.Menu(menubar, tearoff=False)
    menubar.add_cascade(label="Tasks", menu=self.tasks_menu)
    window.configure(menu=menubar)
    window.bind("<Control-q>", lambda event: window.destroy())

  def show(self, lines: list[str], status: str) -> None:
    self.tasks.delete(0, "end")
    self.tasks.insert("end", *lines)
    self.status.configure(text=status)

  def selection(self) -> list[int]:
    return list(self.tasks.curselection())

  def typed_title(self) -> str:
    return self.title.get()

  def clear_entry(self) -> None:
    self.title.set("")

  def chosen_filter(self) -> str:
    return self.which.get()

  def report(self, message: str) -> None:
    messagebox.showerror("Tasks", message)

  def on_add(self, action: Callable[[], None]) -> None:
    self.add.configure(command=action)
    self.entry.bind("<Return>", lambda event: action())

  def on_toggle(self, action: Callable[[], None]) -> None:
    self.toggle.configure(command=action)
    self.tasks.bind("<Double-Button-1>", lambda event: action())

  def on_remove(self, action: Callable[[], None]) -> None:
    self.remove.configure(command=action)

  def on_filter(self, action: Callable[[], None]) -> None:
    for choice in self.filter_choices:
      choice.configure(command=action)

  def on_remove_done(self, action: Callable[[], None]) -> None:
    self.tasks_menu.add_command(label="Remove done tasks", command=action)

### Preview

The first cell writes a small program that shows the view alone, the second one launches it. Make the window bigger: the entry and the list should follow. The "File" › "Quit" menu already closes it; "Tasks" is still empty.

In [ ]:
%%writefile preview.py
import tkinter as tk

from view import TaskView

root = tk.Tk()
root.title("View preview")
view = TaskView(root)
view.pack(fill="both", expand=True)
view.show(["[ ] Buy bread", "[x] Review the course"], "1 task(s) to do")
root.mainloop()

In [ ]:
launch("preview.py")

## 3. The controller

Write `controller.py`: a `Controller` class that receives the model and the view, gives its methods to the view through the `on_…` methods and keeps the view up to date. The controller touches no widget: it only goes through the view's methods, and imports neither tkinter nor `view.py`. The twin lab, with ipywidgets, will reuse this file as is.

- `refresh()` asks the model for the chosen filter's tasks, keeps them in `self.visible`, and shows them, one line per task, as `[ ] title` (to do) or `[x] title` (done), with the number of tasks to do in `status`. Line `i` of the list is task `self.visible[i]`.
- `add()` adds the task typed in. If the model raises a `ValueError`, the view reports its message; otherwise, the entry is cleared and the list refreshed. The view calls it on the "Add" button and on the Return key.
- `toggle()` and `remove()` act on the selected tasks, then refresh the list. The view also calls `toggle()` on a double-click.
- `remove_done()` takes out the done tasks, then refreshes the list. The view calls it from the "Tasks" menu.
- Choosing a filter refreshes the list.

The cell after the solution checks the controller without opening a window.

In [ ]:
# Your code here

### Additional hints

- The controller gives its methods to the view without calling them: `view.on_add(self.add)`.
- It reads the entry and the filter through the view, `self.view.typed_title()` and `self.view.chosen_filter()`, never through its widgets.
- `try` / `except ValueError as error` / `else`, as in the course's controller: the `else` only runs if adding worked.
- The list only shows the current filter's tasks: its line `i` isn't `model.tasks[i]`, hence `self.visible`. A `selected()` method returning `[self.visible[i] for i in self.view.selection()]` serves both `toggle` and `remove`.
- `view` has no type annotation: the controller accepts any view that has these methods, so it imports neither tkinter nor `view.py`.

```python
%%writefile controller.py
from model import Checklist, Task


class Controller:
  def __init__(self, model: Checklist, view) -> None:
    self.model = model
    self.view = view  # any view that has TaskView's methods
    self.visible: list[Task] = []
    view.on_add(self.add)
    ...
    self.refresh()

  def refresh(self) -> None:
    self.visible = self.model.filtered(self.view.chosen_filter())
    lines = []
    for task in self.visible:
      ...
    left = ...
    self.view.show(lines, f"{left} task(s) to do")

  def add(self) -> None:
    try:
      ...
    except ValueError as error:
      ...
    else:
      ...

  def selected(self) -> list[Task]:
    return [self.visible[index] for index in self.view.selection()]

  def toggle(self) -> None: ...

  def remove(self) -> None: ...

  def remove_done(self) -> None: ...
```

### Solution

In [ ]:
%%writefile controller.py
from model import Checklist, Task


class Controller:
  def __init__(self, model: Checklist, view) -> None:
    self.model = model
    self.view = view  # any view that has TaskView's methods
    self.visible: list[Task] = []
    view.on_add(self.add)
    view.on_toggle(self.toggle)
    view.on_remove(self.remove)
    view.on_filter(self.refresh)
    view.on_remove_done(self.remove_done)
    self.refresh()

  def refresh(self) -> None:
    self.visible = self.model.filtered(self.view.chosen_filter())
    lines = []
    for task in self.visible:
      box = "[x]" if task.done else "[ ]"
      lines.append(f"{box} {task.title}")
    left = len(self.model.filtered("to do"))
    self.view.show(lines, f"{left} task(s) to do")

  def add(self) -> None:
    try:
      self.model.add(self.view.typed_title())
    except ValueError as error:
      self.view.report(str(error))
    else:
      self.view.clear_entry()
      self.refresh()

  def selected(self) -> list[Task]:
    return [self.visible[index] for index in self.view.selection()]

  def toggle(self) -> None:
    for task in self.selected():
      self.model.toggle(task)
    self.refresh()

  def remove(self) -> None:
    for task in self.selected():
      self.model.remove(task)
    self.refresh()

  def remove_done(self) -> None:
    self.model.remove_done()
    self.refresh()

### Check without a window

The next cell reloads `controller.py`, then connects it to `FakeView`, a plain-Python view that has `TaskView`'s methods: it keeps what it is shown and the actions the controller gives it, which the cell calls as a click would. It runs everywhere, Colab included: that is what a controller that doesn't import tkinter allows. Expected results: two tasks to do and the duplicate's message, then `[x] Review the course` done, alone in the `done` filter, then `[ ] Buy bread` alone after the removal of "Review the course", and still alone after "Remove done tasks", which takes out "Water the plants".

In [ ]:
import importlib

import controller
import model

importlib.reload(model)
importlib.reload(controller)


class FakeView:
  """TaskView's methods, without a window: it keeps what it is shown."""

  def __init__(self) -> None:
    self.title = ""
    self.which = "all"
    self.chosen: list[int] = []
    self.lines: list[str] = []
    self.status = ""
    self.messages: list[str] = []
    self.actions = {}

  def show(self, lines: list[str], status: str) -> None:
    self.lines, self.status = lines, status

  def selection(self) -> list[int]:
    return self.chosen

  def typed_title(self) -> str:
    return self.title

  def clear_entry(self) -> None:
    self.title = ""

  def chosen_filter(self) -> str:
    return self.which

  def report(self, message: str) -> None:
    self.messages.append(message)

  def on_add(self, action) -> None:
    self.actions["add"] = action

  def on_toggle(self, action) -> None:
    self.actions["toggle"] = action

  def on_remove(self, action) -> None:
    self.actions["remove"] = action

  def on_filter(self, action) -> None:
    self.actions["filter"] = action

  def on_remove_done(self, action) -> None:
    self.actions["remove_done"] = action


view = FakeView()
controller.Controller(model.Checklist(), view)
for title in ("Buy bread", "Review the course", "Buy bread"):
  view.title = title
  view.actions["add"]()
print(view.lines, view.status, view.messages)

view.chosen = [1]
view.actions["toggle"]()
print(view.lines, view.status)

view.which = "done"
view.actions["filter"]()
print(view.lines)
view.chosen = [0]
view.actions["remove"]()
view.which = "all"
view.actions["filter"]()
print(view.lines, view.status)

view.title = "Water the plants"
view.actions["add"]()
view.chosen = [1]
view.actions["toggle"]()
view.actions["remove_done"]()
print(view.lines, view.status)

## 4. The application

Write `app.py`, the only file that puts the three others together: a `main` function that creates the window (with a title and a minimum size), puts the view in it, filling all the space, creates the controller with an empty `Checklist`, then starts the event loop. `app.py` calls it when run as a script.

The cell after the solution launches the application.

In [ ]:
# Your code here

### Additional hints

- `root.minsize(width, height)` sets the minimum size; `view.pack(fill="both", expand=True)` gives all the space to the view.
- `if __name__ == "__main__":` at the end of the file: `main()` doesn't run when another module imports `app`.

```python
%%writefile app.py
import tkinter as tk

from controller import Controller
from model import Checklist
from view import TaskView


def main() -> None:
  root = tk.Tk()
  ...
  root.mainloop()


if __name__ == "__main__":
  main()
```

### Solution

In [ ]:
%%writefile app.py
import tkinter as tk

from controller import Controller
from model import Checklist
from view import TaskView


def main() -> None:
  root = tk.Tk()
  root.title("Tasks")
  root.minsize(380, 300)
  view = TaskView(root)
  view.pack(fill="both", expand=True)
  Controller(Checklist(), view)
  root.mainloop()


if __name__ == "__main__":
  main()

### Launch

Try it: add tasks with the button and with Return, then an empty title and a duplicate; toggle several tasks at once (Ctrl+click) and another one with a double-click; change the filter; remove some; clear the done tasks from the "Tasks" menu, then quit with Ctrl+Q. The ipywidgets twin lab reuses `model.py` and `controller.py` without changing a line: only the view changes.

In [ ]:
launch("app.py")